# Embedding basics

Polymarket's own search matches words. Embedding search matches meaning, which
is what you want when you know the question you are asking but not the phrasing
the market used.

This notebook needs the optional extra and a key:

```shell
pip install pyolymarket[embeddings]
export PYOLY_OPENAI_API_KEY=...
```

It ships without outputs for that reason. Any OpenAI-compatible endpoint will
do, including a local one, as shown near the end.

In [ ]:
import sys
from pathlib import Path

# Run straight from a clone, without installing the package first.
src = Path.cwd().parent / "src"
if src.is_dir():
    sys.path.insert(0, str(src))

import os

import numpy as np
import pandas as pd

import pyolymarket as pyoly

print("key found:", bool(os.getenv(pyoly.config.EMB_API_KEY_ENV)))

## Embedding text

`embed` takes one string and returns one vector. The model is whatever
`config.embedding_model` names.

In [ ]:
print(pyoly.config.embedding_model)

vector = pyoly.embed("Will the Federal Reserve cut rates in September?")
print(len(vector), "dimensions")
print(vector[:5])

`embed_list` takes many strings in one request and also hands back the token
count, which is the part you get billed on. Requests longer than 2048 strings
are split and recombined for you.

In [ ]:
questions = [
    "Will the Federal Reserve cut rates in September?",
    "Will the Fed lower interest rates this fall?",
    "Will Manchester City win the Premier League?",
]

vectors, tokens = pyoly.embed_list(questions)
print(len(vectors), "vectors,", tokens, "tokens")

Cosine similarity between those three shows why this is worth doing: the two
rate questions share no keywords beyond "rates", but they land next to each
other.

In [ ]:
matrix = np.array(vectors)
normalized = matrix / np.linalg.norm(matrix, axis = 1, keepdims = True)

pd.DataFrame(normalized @ normalized.T,
             index = ["fed cut", "fed lower", "football"],
             columns = ["fed cut", "fed lower", "football"]).round(3)

## Embedding an event

`Event.vectorize()` embeds the first line of the event's description, falling
back to the whole description when that line is empty, and stores the result
on `.vec`.

In [ ]:
event = pyoly.polymarket_search_event("federal reserve", results = 1)

print(event.title)
print(event.data["description"].split("\n")[0])

In [ ]:
event.vectorize()
print(len(event.vec), "dimensions stored on event.vec")

## Searching by meaning

`embedded_search_event` ranks a table of pre-embedded events against a query
by cosine similarity. The table needs an `id` column and an `embeddedVector`
column of equal-length vectors, and here we build a small one by hand.

In [ ]:
events = pyoly.polymarket_search_event("federal reserve interest rates", results = 10)


def first_line(event):
    """What the cacher embeds: the first line, or the whole description when
    that line is empty."""
    description = event.data.get("description") or ""
    return description.split("\n")[0] or description or event.title


vectors, tokens = pyoly.embed_list([first_line(e) for e in events])
print(tokens, "tokens for", len(events), "events")

corpus = pd.DataFrame({"id": [e.id for e in events], "embeddedVector": vectors})
corpus.head()

In [ ]:
hits = pyoly.embedded_search_event("will borrowing get cheaper", results = 3,
                                   data = corpus)

for hit in hits:
    print(hit.title)

With `results=1` you get a bare `Event` rather than a list, which matches how
the other search helpers in this library behave.

In [ ]:
pyoly.embedded_search_event("who chairs the Fed", data = corpus)

A table missing either required column is rejected up front, before any
embedding request is paid for.

In [ ]:
try:
    pyoly.embedded_search_event("anything", data = pd.DataFrame({"id": ["1"]}))
except pyoly.EmbeddingError as error:
    print(error)

## The disk cache

Building that table by hand is fine for a handful of events. For the catalogue
there is `pyoly.cacher`, which walks every open market on Gamma, embeds each
new event it has not seen before, and writes the result next to your working
directory.

It is not run here: a full pass pulls thousands of markets and pays for every
new embedding. The shape of it is:

```python
pyoly.config.caching = True          # "on", or "csv" / "npy" for one format
pyoly.config.log = True              # progress bar and INFO logging

pyoly.cacher.cache()                 # build it
pyoly.cacher.load()                  # or read an existing one off disk

pyoly.cacher.events                  # one row per event, with embeddedVector
pyoly.cacher.markets                 # one row per market, joined to its event
pyoly.cacher.events_vecs             # the embedding matrix, row-aligned
```

Re-running `cache()` only embeds events that are new since the last pass, so
a refresh is much cheaper than the first build.

In [ ]:
print(pyoly.config.cache_level)
print(pyoly.config.CACHE_DIR)

Once a cache exists, `embedded_search_event` finds it on its own and the
`data=` argument becomes unnecessary. With caching off and no table passed, it
tells you which of the two is missing rather than searching nothing:

In [ ]:
try:
    pyoly.embedded_search_event("rate cuts")
except pyoly.EmbeddingError as error:
    print(error)

## Pointing somewhere else

`config.embedding_base_url` accepts any OpenAI-compatible endpoint, and a
local one needs no key at all. The client is rebuilt when this changes, so you
can switch mid-session.

In [ ]:
pyoly.config.embedding_base_url = "http://localhost:11434/v1"
pyoly.config.embedding_model = "nomic-embed-text"

print(pyoly.config.emb_api_key)